# Petta AI Care Assistant — Qwen2.5-3B-Instruct QLoRA Fine-Tuning

Fine-tunes **`Qwen/Qwen2.5-3B-Instruct`** on Google Colab (T4 / L4 / A100 GPU) or Kaggle (T4 x2 / P100) using 4-bit QLoRA (`bitsandbytes` + `peft` + `trl`) with:
- Veterinary pet-care Q&A (`humane-lab/PetQA`, `viggovet/Veterinary-Med`)
- Filipino / Taglish grounding (`rhyliieee/tagalog-filipino-english-translation`)
- Petta Taglish, Bikol/Cebuano, & Emergency Escalation Guardrails

**Before running:** Go to `Runtime → Change runtime type → GPU` (T4 or L4).

In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece protobuf

## 1. Prepare Dataset
Clones `https://github.com/suikuroMi/Petta.git` and runs `ai/prepare_dataset.py`.

In [ ]:
%cd /content
!rm -rf Petta
!git clone https://github.com/suikuroMi/Petta.git
%cd Petta
!python ai/prepare_dataset.py --out-dir ai/data --max-vet 3500 --max-filipino 1000

## 2. Load `Qwen/Qwen2.5-3B-Instruct` in 4-bit QLoRA

In [ ]:
import inspect
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./petta-qwen2.5-3b-lora"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
if not use_bf16:
    for param in model.parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float32 if param.requires_grad else torch.float16)
model.config.use_cache = False
print(f"Loaded {BASE_MODEL} with dtype={compute_dtype}, bf16={use_bf16}")

## 3. Load Train/Eval JSONL & Configure LoRA

In [ ]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "ai/data/petta_train.jsonl",
        "eval": "ai/data/petta_eval.jsonl",
    },
)

def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
eval_ds = dataset["eval"].map(format_chat, remove_columns=dataset["eval"].column_names)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(f"Train rows: {len(train_ds)}, Eval rows: {len(eval_ds)}")

## 4. Train with `SFTTrainer`

In [ ]:
import inspect
sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 20,
    "logging_steps": 25,
    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_steps": 200,
    "fp16": False,
    "bf16": use_bf16,
    "optim": "paged_adamw_8bit",
    "dataset_text_field": "text",
    "report_to": "none",
}
if "max_length" in sft_params:
    sft_kwargs["max_length"] = 1024
elif "max_seq_length" in sft_params:
    sft_kwargs["max_seq_length"] = 1024

training_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)

## 5. Test Taglish & Emergency Guardrail Inference

In [ ]:
from ai.prepare_dataset import PETTA_SYSTEM_PROMPT

gen_model = trainer.model
gen_model.config.use_cache = True
gen_model.eval()

def ask_petta(prompt_text: str) -> str:
    messages = [
        {"role": "system", "content": PETTA_SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("--- TAGLISH CARE ---")
print(ask_petta("Pwede ba magbigay ng kanin at manok sa aso kong may konting diarrhea?"))
print("\n--- EMERGENCY TRIAGE ---")
print(ask_petta("Nangingisay yung aso ko tapos nakakain ng dark chocolate!"))

## 6. Push LoRA Adapter to Hugging Face Hub (Free)
Create a free Hugging Face Access Token (`Write`) at https://huggingface.co/settings/tokens and paste it when prompted.

In [ ]:
from huggingface_hub import login
login()

HF_REPO_ID = "YOUR_HF_USERNAME/petta-qwen2.5-3b-lora"  # <-- change this
trainer.model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print("Pushed to https://huggingface.co/" + HF_REPO_ID)